##### Findings Summary
_Full exploration process and evidence in `eda.ipynb`._

- `AGMT_END_YMD` (date column) mixed with non-date text (`'무약정'`, `'정보없음'`)
- Count columns (`TV_SCRB`, `ANALOG_SCRB`, etc.) contain literal `'\"\"'` placeholder values
- Categorical values are consistent across all 4 quarters, except `AGE_GRP10='10대미만'` (6 rows) — safe to combine the files
- `CH_25_RATIO_MONTH1` has 10 rows exceeding the theoretical max of 100; confirmed via `TOTAL_USED_DAYS` as a new-customer edge case (insufficient history for the 3-month-average calc)
- `DIGITAL_SCRB` / `TV_I_CNT` have unusually large max values, suggesting a bulk/business account (e.g. an apartment complex) subscribed under one record
- Target variable (`cancel_yn`) is imbalanced: 94.3% 유지 (retained) vs 5.7% 해지 (cancelled)
- Most customers (about 97.6%, 1,950,872) appear in all 4 quarters; the rest appear in only 1-3, relevant for train/test splitting later
- 33 rows show simultaneous `'\"\"'`/`'기타'`/`'정보없음'` sentinels across ~9 columns (anchored on `STB_RES_1M_YN`) — determined to be internal test/special accounts rather than real customers, excluded
- `sha2_hash + p_mt` duplicates (3.67% of rows, 419,937 excess rows): NOT exact-copy duplicates — the only column that differs between the two rows is `cancel_yn` itself. Affects 40,063 customers (1.84% of all customers), and once a customer is affected, every subsequent month they appear is also duplicated (never reverts to a single row) until they drop out of the dataset — a persistent structural conflict, not a one-time snapshot glitch. Bulk/business accounts (`TV_I_CNT` ≥ 10) are ~2x more common among affected customers than normal ones (1.58% vs 0.78% of rows) — a real but partial contributing factor, since 98%+ of affected customers still have ordinary-sized accounts and are unexplained by it. Root cause otherwise undetermined (likely a pipeline/join issue upstream); since there's no reliable signal to decide which `cancel_yn` value is correct for any of them, these 40,063 customers are excluded entirely below

##### Data Cleaning
- Drop 33 corrupted rows (multiple columns showed literal `""` simultaneously, including STB_RES_1M_YN) and 6 `AGE_GRP10='10대미만'` rows (too small a sample to be meaningful)
- Drop all rows belonging to the 40,063 customers with persistent `sha2_hash + p_mt` / `cancel_yn` conflicts — no reliable way to determine which `cancel_yn` value is correct for them
- `AGMT_KIND_NM` / `AGMT_END_YMD`: convert `'정보없음'` (no info) to NULL; `'무약정'` (no fixed contract) is kept as-is, it is a valid category, not missing
- `CH_25_RATIO_MONTH1`: cap values above 100 at 100 (physically impossible ratio, likely a new-customer edge case in the 3-month-average calc)
- `PROD_NM_GRP`'s `'기타'` (108 rows): kept as its own category (no transformation) — tree models can split on it natively without needing an ordinal position

In [1]:
import duckdb
import pandas as pd
import os

# The C: drive is almost completely full (only ~6.8GB free) — that's the real
# root cause of the earlier errors, not just a DuckDB setting. Process the 4
# files ONE AT A TIME instead of combined (each is ~1/4 the size, so peak
# memory/disk needs drop proportionally) and write each to its own Parquet
# file. read_parquet() can read multiple files as one table afterward.
temp_dir = os.path.expanduser("~/AppData/Local/duckdb_tmp")
os.makedirs(temp_dir, exist_ok=True)

con = duckdb.connect(config={
    "temp_directory": temp_dir,
    "memory_limit": "4GB",
    "threads": "2",
})
con.sql("PRAGMA max_temp_directory_size='2GiB'")

files = [
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202302_to_202304 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202305_to_202307 (2).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202308_to_202310 (1).csv",
    "LGHelloVision Data/250207 3기 데이터 (추가 2)/sha_tps_cancel_202311_to_202312 (1).csv",
]
file_list_sql = ", ".join(f"'{f}'" for f in files)

# Step 1: figure out which customers conflict, reading only the 2 columns
# needed for that (much lighter than scanning all 38 columns)
con.sql(f"""
    CREATE OR REPLACE TABLE conflicted_customers AS
    SELECT DISTINCT sha2_hash
    FROM (
        SELECT sha2_hash, p_mt
        FROM read_csv([{file_list_sql}], all_varchar=true, union_by_name=true)
    )
    GROUP BY sha2_hash, p_mt
    HAVING count(*) > 1
""")
n_conflicted = con.sql("SELECT count(*) AS n FROM conflicted_customers").df()["n"][0]
print(f"conflicted customers: {n_conflicted:,}")

# Step 2: clean + write each file separately, in its own smaller Parquet part
out_files = []
for i, f in enumerate(files, start=1):
    out_path = f"df_clean_part{i}.parquet"
    con.sql(f"""
        COPY (
            SELECT
                * EXCLUDE (AGMT_KIND_NM, AGMT_END_YMD, CH_25_RATIO_MONTH1),
                CASE WHEN AGMT_KIND_NM = '정보없음' THEN NULL ELSE AGMT_KIND_NM END AS AGMT_KIND_NM,
                CASE WHEN AGMT_END_YMD = '정보없음' THEN NULL ELSE AGMT_END_YMD END AS AGMT_END_YMD,
                LEAST(TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE), 100) AS CH_25_RATIO_MONTH1
            FROM read_csv(['{f}'], all_varchar=true, union_by_name=true)
            WHERE STB_RES_1M_YN != '""'
              AND AGE_GRP10 != '10대미만'
              AND sha2_hash NOT IN (SELECT sha2_hash FROM conflicted_customers)
        ) TO '{out_path}' (FORMAT PARQUET)
    """)
    out_files.append(out_path)
    print(f"wrote {out_path}")

row_count = con.sql(f"SELECT count(*) AS n FROM read_parquet({out_files})").df()
print(f"rows: {row_count['n'][0]:,}")
con.sql(f"SELECT * FROM read_parquet({out_files}) LIMIT 5").df()

conflicted customers: 40,063
wrote df_clean_part1.parquet
wrote df_clean_part2.parquet
wrote df_clean_part3.parquet
wrote df_clean_part4.parquet
rows: 22,053,564


,sha2_hash,SVC_USE_DAYS_GRP,MEDIA_NM_GRP,PROD_NM_GRP,PROD_OLD_YN,PROD_ONE_PLUS_YN,STB_RES_1M_YN,SVOD_SCRB_CNT_GRP,PAID_CHNL_CNT_GRP,SCRB_PATH_NM_GRP,...,CH_25_RATIO_MEAN_3MM,CH_FAV_RNK1,KIDS_USE_PV_MONTH1,NFX_USE_YN,YTB_USE_YN,p_mt,cancel_yn,AGMT_KIND_NM,AGMT_END_YMD,CH_25_RATIO_MONTH1
0,6c2fb1fb0b1e316975157671e03d0e2eb3d250a7373321...,6개월미만,UHD,이코노미,N,N,N,0건,0건,직영몰,...,3.63,기타,2.0,N,N,202303,유지,신규,20260213,3.63
1,39b9ff560dbcbe4e138f04fd082d55d921c5528f1cc279...,6개월미만,UHD,베이직,N,N,N,0건,0건,현장경로,...,5.67,MBC,1.0,N,N,202303,유지,신규,20260213,5.67
2,db168b085b5c0cd2d21faab70ceb4ec36c5409aea1f8f4...,6개월미만,UHD,베이직,N,N,N,0건,0건,현장경로,...,0.36,기타,0.0,N,N,202303,유지,신규,20260214,0.36
3,59a02dfe6d4e5e4ffc1b295dc0ebf0fdf7ffc37321f653...,6개월미만,HD,이코노미,N,Y,N,0건,0건,I/B,...,1.85,SBS,0.0,N,N,202303,유지,신규,20260214,1.85
4,6f415993f9b6968f3b7bd8b2188f8651dfdc7e50e3022a...,6개월미만,UHD,베이직,N,N,N,0건,0건,현장경로,...,0.55,연합뉴스TV,0.0,N,Y,202303,유지,신규,20260216,0.55


##### Verify Cleaning Result
- Re-run the same `SUMMARIZE` profile directly on `df_clean` (DuckDB queries the in-memory pandas DataFrame directly via its replacement scan, no re-read of the CSVs needed)
- Compare each column's min/max/approx_unique/null_percentage against the original profile above to confirm the cleaning worked as expected

In [2]:
profile_clean = duckdb.sql("""
    SELECT column_name, min, max, approx_unique, null_percentage
    FROM (SUMMARIZE SELECT * EXCLUDE (sha2_hash) FROM read_parquet('df_clean_part*.parquet'))
""").df()

profile_clean

,column_name,min,max,approx_unique,null_percentage
0,SVC_USE_DAYS_GRP,12개월~24개월미만,6개월미만,5,0.0
1,MEDIA_NM_GRP,HD,기타,3,0.0
2,PROD_NM_GRP,기타,프리미엄,6,0.0
3,PROD_OLD_YN,N,Y,2,0.0
4,PROD_ONE_PLUS_YN,N,Y,2,0.0
5,STB_RES_1M_YN,N,Y,2,0.0
6,SVOD_SCRB_CNT_GRP,0건,3건 이상,4,0.0
7,PAID_CHNL_CNT_GRP,0건,3건 이상,4,0.0
8,SCRB_PATH_NM_GRP,I/B,현장경로,9,0.0
9,INHOME_RATE,0.0,알수없음,13,0.0


##### Verify Categorical Values After Cleaning
- Re-run the same UNPIVOT value-count check (from `column_check.ipynb`) on `df_clean` this time
- Confirm that `'""'`, `'기타'`, `'정보없음'` sentinel values are gone (or reduced as expected) from each column

In [3]:
categorical_cols = [
    "SVC_USE_DAYS_GRP", "MEDIA_NM_GRP", "PROD_NM_GRP", "PROD_OLD_YN", "PROD_ONE_PLUS_YN",
    "AGMT_KIND_NM", "STB_RES_1M_YN", "SVOD_SCRB_CNT_GRP", "PAID_CHNL_CNT_GRP", "SCRB_PATH_NM_GRP",
    "INHOME_RATE", "AGMT_END_SEG", "BUNDLE_YN", "DIGITAL_GIGA_YN", "DIGITAL_ALOG_YN",
    "CH_LAST_DAYS_BF_GRP", "VOC_TOTAL_MONTH1_YN", "VOC_STOP_CANCEL_MONTH1_YN", "AGE_GRP10",
    "EMAIL_RECV_CLS_NM", "SMS_SEND_CLS_NM", "NFX_USE_YN", "YTB_USE_YN", "cancel_yn", "CH_FAV_RNK1",
]
unpivot_cols_sql = ", ".join(categorical_cols)

value_check_clean = duckdb.sql(f"""
    UNPIVOT read_parquet('df_clean_part*.parquet')
    ON {unpivot_cols_sql}
    INTO
        NAME column_name
        VALUE value
""")

duckdb.sql("""
    SELECT column_name, value, count(*) AS n
    FROM value_check_clean
    WHERE value IN ('""', '기타', '정보없음', '10대미만')
    GROUP BY column_name, value
    ORDER BY column_name, n DESC
""")

┌──────────────────┬──────────┬──────────┐
│   column_name    │  value   │    n     │
│     varchar      │ varchar  │  int64   │
├──────────────────┼──────────┼──────────┤
│ AGMT_END_SEG     │ 기타     │       33 │
│ CH_FAV_RNK1      │ 기타     │ 13541647 │
│ MEDIA_NM_GRP     │ 기타     │    71115 │
│ PROD_NM_GRP      │ 기타     │       75 │
│ SCRB_PATH_NM_GRP │ 정보없음 │    39254 │
│ SCRB_PATH_NM_GRP │ 기타     │    39192 │
└──────────────────┴──────────┴──────────┘

##### Find Columns That Still Need Cleaning
- The check above only looks for the 4 sentinel values we already knew about. Broaden it: scan every categorical column for anything that *looks* like a missing/unknown marker (`없음`, `모름`, `미상`, `unknown`, `기타`, `""`), not just the ones already on our radar
- Separately, check columns that are supposed to be numeric for leftover non-numeric strings (this is how `INHOME_RATE`'s `'알수없음'` turned up — it's a numeric-scale column but never got cast/cleaned like `CH_25_RATIO_MONTH1` did)

In [4]:
# Broad sentinel-pattern scan across all categorical columns
broad_sentinel_check = duckdb.sql("""
    SELECT column_name, value, count(*) AS n
    FROM value_check_clean
    WHERE value = '""'
       OR value LIKE '%없음%'
       OR value LIKE '%모름%'
       OR value LIKE '%미상%'
       OR lower(value) LIKE '%unknown%'
       OR value = '기타'
    GROUP BY column_name, value
    ORDER BY column_name, n DESC
""").df()

broad_sentinel_check

,column_name,value,n
0,AGE_GRP10,연령없음,62430
1,AGMT_END_SEG,기타,33
2,CH_FAV_RNK1,기타,13541647
3,CH_LAST_DAYS_BF_GRP,3개월내없음,3629629
4,INHOME_RATE,알수없음,3030808
5,MEDIA_NM_GRP,기타,71115
6,PROD_NM_GRP,기타,75
7,SCRB_PATH_NM_GRP,정보없음,39254
8,SCRB_PATH_NM_GRP,기타,39192


##### Fix: SCRB_PATH_NM_GRP = '정보없음' → NULL
- Same treatment as `AGMT_KIND_NM`/`AGMT_END_YMD`: `'정보없음'` means "subscription path unknown", a missing value, not a valid category like `'기타'`
- Patch the already-saved Parquet parts in place instead of re-reading the raw CSVs (avoids redoing the expensive full pipeline)

In [5]:
import glob

for path in sorted(glob.glob("df_clean_part*.parquet")):
    tmp_path = path + ".tmp"
    duckdb.sql(f"""
        COPY (
            SELECT
                * EXCLUDE (SCRB_PATH_NM_GRP),
                CASE WHEN SCRB_PATH_NM_GRP = '정보없음' THEN NULL ELSE SCRB_PATH_NM_GRP END AS SCRB_PATH_NM_GRP
            FROM read_parquet('{path}')
        ) TO '{tmp_path}' (FORMAT PARQUET)
    """)
    os.replace(tmp_path, path)
    print(f"patched {path}")

duckdb.sql("""
    SELECT SCRB_PATH_NM_GRP, count(*) AS n
    FROM read_parquet('df_clean_part*.parquet')
    GROUP BY SCRB_PATH_NM_GRP
    ORDER BY n DESC
""").df()

patched df_clean_part1.parquet
patched df_clean_part2.parquet
patched df_clean_part3.parquet
patched df_clean_part4.parquet


,SCRB_PATH_NM_GRP,n
0,현장경로,8225308
1,O/B,5453621
2,I/B,4288227
3,일반상담,3094858
4,직영몰,778180
5,임직원,102930
6,None,39254
7,기타,39192
8,전략채널,31208
9,렌탈제휴,786


##### Fix: INHOME_RATE = '알수없음' → NULL, cast to numeric
- Unlike `기타`, `'알수없음'` means "in-home rate unknown" — a missing value, not a valid category
- This column is meant to be numeric (0-100 scale), so also cast it to `DOUBLE` while fixing it (it was left as a string this whole time, unlike `CH_25_RATIO_MONTH1` which was already cast during the main pipeline)
- Patch the Parquet parts in place, same as the `SCRB_PATH_NM_GRP` fix above

In [4]:
import duckdb
import glob
import os

for path in sorted(glob.glob("df_clean_part*.parquet")):
    tmp_path = path + ".tmp"
    duckdb.sql(f"""
        COPY (
            SELECT
                * EXCLUDE (INHOME_RATE),
                CASE WHEN INHOME_RATE = '알수없음' THEN NULL ELSE TRY_CAST(INHOME_RATE AS DOUBLE) END AS INHOME_RATE
            FROM read_parquet('{path}')
        ) TO '{tmp_path}' (FORMAT PARQUET)
    """)
    os.replace(tmp_path, path)
    print(f"patched {path}")

duckdb.sql("""
    SELECT INHOME_RATE, count(*) AS n
    FROM read_parquet('df_clean_part*.parquet')
    GROUP BY INHOME_RATE
    ORDER BY n DESC
""").df()

patched df_clean_part1.parquet
patched df_clean_part2.parquet
patched df_clean_part3.parquet
patched df_clean_part4.parquet


,INHOME_RATE,n
0,10.0,3905474
1,0.0,3540010
2,20.0,3469282
3,NaN,3030808
4,30.0,2891071
5,40.0,2232435
6,50.0,1561090
7,60.0,904906
8,70.0,380620
9,80.0,117562


In [6]:
import duckdb

# Numeric-scale columns: find non-numeric strings hiding in them
numeric_cols = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MONTH1", "CH_25_RATIO_MEAN_3MM",
    "KIDS_USE_PV_MONTH1", "INHOME_RATE",
]
unpivot_num_cols_sql = ", ".join(numeric_cols)
# INHOME_RATE/CH_25_RATIO_MONTH1 are now DOUBLE (already cleaned) while the
# rest are still VARCHAR — UNPIVOT needs matching types, so cast everything
# to VARCHAR first in a subquery before unpivoting
cast_cols_sql = ", ".join(f"CAST({c} AS VARCHAR) AS {c}" for c in numeric_cols)

value_check_numeric = duckdb.sql(f"""
    UNPIVOT (
        SELECT {cast_cols_sql}
        FROM read_parquet('df_clean_part*.parquet')
    )
    ON {unpivot_num_cols_sql}
    INTO
        NAME column_name
        VALUE value
""")

non_numeric_check = duckdb.sql("""
    SELECT column_name, value, count(*) AS n
    FROM value_check_numeric
    WHERE value IS NOT NULL AND TRY_CAST(value AS DOUBLE) IS NULL
    GROUP BY column_name, value
    ORDER BY column_name, n DESC
""").df()

non_numeric_check

,column_name,value,n


##### KIDS_USE_PV_MONTH1: How Skewed Is It?
- Already flagged in `column_check.ipynb` as heavily right-skewed (mean 0.34-0.43 vs median 0). Look at the tail (p90/p99/p999) and % of rows at exactly 0 to see how extreme it actually is before deciding what (if anything) to do about it
- **Result**: 86.03% zero, p90=1.0, p99=5.73, p999=40.63, max=4621.0 — "mostly unused, a few heavy users" pattern (not a physically-impossible value like `CH_25_RATIO_MONTH1`'s >100 case)
- **Decision: no treatment needed.** Tree models (LightGBM/XGBoost) don't assume normality, so raw skew isn't a problem the way it would be for a linear model — no log transform or capping applied

In [2]:
import duckdb
import pandas as pd
kids_use_pv_month1_summary = duckdb.sql("""
    SELECT
        count(*) AS n,
        min(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE)) AS min_v,
        avg(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE), 0.5) AS median_v,
        approx_quantile(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE), 0.9) AS p90,
        approx_quantile(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE), 0.99) AS p99,
        approx_quantile(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE), 0.999) AS p999,
        max(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE)) AS max_v,
        count(*) FILTER (WHERE TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE) = 0) AS n_zero,
        round(count(*) FILTER (WHERE TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE) = 0) * 100.0 / count(*), 2) AS pct_zero
    FROM read_parquet('df_clean_part*.parquet')
""").df()

kids_use_pv_month1_summary

,n,min_v,avg_v,median_v,p90,p99,p999,max_v,n_zero,pct_zero
0,22053564,0.0,0.396931,0.0,1.0,5.732057,40.627943,4621.0,18972406,86.03


##### Fix: Cast Remaining Numeric Columns from VARCHAR to DOUBLE
- The CSVs were read with `all_varchar=true` from the start (in `clean-pipeline-01`) so every categorical column could be scanned the same way for hidden sentinel values (`정보없음`, `알수없음`, etc.) — this is how `INHOME_RATE`/`SCRB_PATH_NM_GRP` were caught
- Side effect: this got baked into the saved Parquet files. `CH_25_RATIO_MONTH1`/`INHOME_RATE` were already cast to `DOUBLE` during cleaning, but 10 other numeric-scale columns are still stored as `VARCHAR` — fine for `TRY_CAST(...)` on the fly during EDA, but tree models need real numeric dtypes at training time
- Patch the Parquet parts in place, casting all 10 remaining numeric columns to `DOUBLE` at once

In [ ]:
import duckdb
import glob
import os

numeric_cols_to_cast = [
    "TOTAL_USED_DAYS", "TV_SCRB", "ANALOG_SCRB", "DIGITAL_SCRB",
    "TOTAL_INTERNET_SCRB", "GIGA_INTERNET_SCRB", "TV_I_CNT",
    "CH_HH_AVG_MONTH1", "CH_25_RATIO_MEAN_3MM", "KIDS_USE_PV_MONTH1",
]
exclude_sql = ", ".join(numeric_cols_to_cast)
cast_select_sql = ",\n                ".join(
    f"TRY_CAST({c} AS DOUBLE) AS {c}" for c in numeric_cols_to_cast
)

for path in sorted(glob.glob("df_clean_part*.parquet")):
    tmp_path = path + ".tmp"
    duckdb.sql(f"""
        COPY (
            SELECT
                * EXCLUDE ({exclude_sql}),
                {cast_select_sql}
            FROM read_parquet('{path}')
        ) TO '{tmp_path}' (FORMAT PARQUET)
    """)
    os.replace(tmp_path, path)
    print(f"patched {path}")

duckdb.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('df_clean_part*.parquet'))
    WHERE column_name IN ('TOTAL_USED_DAYS', 'TV_SCRB', 'ANALOG_SCRB', 'DIGITAL_SCRB',
        'TOTAL_INTERNET_SCRB', 'GIGA_INTERNET_SCRB', 'TV_I_CNT', 'CH_HH_AVG_MONTH1',
        'CH_25_RATIO_MEAN_3MM', 'KIDS_USE_PV_MONTH1', 'CH_25_RATIO_MONTH1', 'INHOME_RATE')
""").df()

##### Cleaned Data Already Saved
- `clean-pipeline-01` processes and saves each quarterly file separately (`df_clean_part1.parquet` ... `df_clean_part4.parquet`) — done to keep peak memory/disk usage low on a machine with very little free disk space
- `read_parquet('df_clean_part*.parquet')` reads all 4 as one logical table, so downstream code doesn't need to care that it's split
- Sanity check: confirm the files exist and report their sizes

In [ ]:
import os
import glob

for path in sorted(glob.glob("df_clean_part*.parquet")):
    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"{path}: {size_mb:,.1f} MB")